## **一、S7｜把训练变成一项可解释、可恢复的实验**

本篇合并 S7-01「训练系统性能与公平架构实验」和 S7-02「MiniGen-Bench 评测与消融」。S4–S6 已经回答模型怎样预测速度；这里学习如何在真实数据上组织优化、验证、采样和比较，最终说明 **模型学到了什么、花了多少资源、结论能支持到哪里**。

阅读路线是「冻结实验问题 → 检查数据与接口 → 完成一次可靠更新 → 短跑排错 → 测量成本 → B0 正式训练 → B1/B2 配对实验 → 冻结评测 → 解释结果」。本篇给工程参考函数、执行清单和独立算术检查，不安排 toy 模型训练。你随后把这些约定接入项目的正式训练器。

[学习目录](../MiniGen2学习目录.md) · [项目计划](../MiniGen2重构规划.md) · [当前交接](../../reports/v2/00_handoff.md) · [S4 Flow](S4_01_FlowMatching原理与ODE采样.ipynb) · [S5 单流](S5_02_SingleStreamDiT架构与实现.ipynb) · [S6 混合主干](S6_01_HybridDiT双流单流架构.ipynb)

### **1. 当前起点和本课交付**

2026 年 11 月 8 日读取的项目状态是：P2 已验收，停在 P3-01。ArtBench v1 共 46,298 张，train 37,042、validation 4,637、test 4,619；正式缓存覆盖 train/validation，测试图保持封存。冻结 VAE 的细线、文字、小物体重建损失已记为表示局限。Caption 审核的 240 对里有 71 条明确事实错误，这个 **审计样本比例不能直接当成全量错误率**；正式条件训练前需要确定修订、筛除或接受噪声的策略。

当前 CLI 的 `train/sample/evaluate/cache` 是占位入口。P2 的专用缓存脚本已经实现，两者并不冲突。本课的训练函数是 **Notebook 中可阅读的工程参考**，没有替项目完成 P3，也没有执行云训练。学习参考定义、独立数值检查、你的理解验收和正式训练结果分别记录。

| 要完成的工作 | 为什么做 | 进入下一步需要的证据 |
| --- | --- | --- |
| 冻结问题、数据和评测协议 | 防止训练完再改比较标准 | 配置、数据/组件/Caption/Prompt 身份 |
| 验证更新、恢复和条件链路 | 确认下降的损失对应预期计算 | 数值、梯度、同序恢复和条件干预 |
| B0 短跑与 GPU pilot | 找质量瓶颈并估真实资源 | 固定图库、性能记录、费用估计 |
| B0 长训，随后 B1/B2 | 每次只增加一个可诊断环节 | 模型规格、真实训练曲线和检查点 |
| 验证选型后统一测试 | 降低重复选参数的偏差 | 冻结配方、全量生成清单和失败统计 |
| 写结论 | 让工程结果回答研究问题 | 逐 run 结果、区间、成本和适用边界 |

本课的小型检查与 HTML 预览统一放 `artifacts/learning/s7/`。正式训练建议独立使用 `artifacts/runs/<run_id>/`，后续由工程阶段实现；原始学习数据不混入 ArtBench。

## **二、先定义训练究竟要学什么**

### **1. 三个模型共享同一个学习任务**

设 $z$ 是已经按组件规则缩放的干净潜变量，$\epsilon\sim\mathcal N(0,I)$，$t\sim U(0,1)$，$c$ 为文本 token 特征及掩码。基线使用独立噪声配对的线性路径：

$$z_t=(1-t)\epsilon+t z,\qquad u=z-\epsilon.$$

$$L(\theta)=\mathbb E_{z,c,\epsilon,t}\left[\frac{1}{D}\lVert v_\theta(z_t,t,c)-u\rVert^2\right],\qquad D=c_z h_z w_z.$$

训练输入只有 $z_t,t,c$；干净 $z$ 和噪声 $\epsilon$ 用来构造标签，不能额外喂给主干。每次更新重新抽噪声、时间和条件丢弃；固定监督只用于早期排错。B0/B1/B2 共享目标、表示和采样协议，改变图文交互的主干。

平方误差在无限数据和足够模型能力下的最优预测是 $\mathbb E[u\mid z_t,t,c]$。模型面对多条可能经过同一位置的配对路径，学习其条件平均速度；采样沿学到的场积分，所以局部速度 MSE 与最终图片质量并非同一个量。时间采样分布、损失权重或路径一旦更改，就改变了学习问题，应作为单独配方登记。 [Flow Matching 论文第 3 节](https://arxiv.org/abs/2210.02747)

### **2. 从正式缓存追踪一批张量**

| 阶段 | 形状 | 必须核对 |
| --- | --- | --- |
| 缓存潜变量 $z$ | `[b,4,32,32]` | 后验均值已缩放，不能再次乘 scale |
| 缓存文本 $c$ | `[b,77,512]` | last_hidden_state，不能换成 pooled 向量 |
| 文本掩码 | `[b,77]` bool | True 为 BOS/内容/EOS，False 为右侧 padding |
| 随机时间 | `[b]` FP32 | 广播成 `[b,1,1,1]`，特征刻度与积分时间分开 |
| 噪声、插值、速度目标 | `[b,4,32,32]` | 有限、同设备，构造目标优先 FP32 |
| 主干图像 token | `[b,256,d]` | patch=2，$16\times16=256$ |
| 预测、逐样本误差 | `[b,4,32,32]`、`[b]` | 先对潜变量元素平均，再对样本平均 |

磁盘 latent/text 是 FP16，以节约缓存体积；编码器前向使用经过验收的 FP32。训练时把读取的张量转 FP32，再让 autocast 选择算子的计算精度。转回 FP32 不会恢复 FP16 存储时丢掉的精度，这项误差已属于缓存验收的一部分。组件 revision、缩放、编码策略与预处理身份须一致。

现有 `RepresentationCache` 构造器核对期待身份、回执、文件哈希和样本顺序；`split("train")`、`split("validation")` 返回数据集；`blank()` 给独立空文本及掩码。应由当前源重新派生期待身份，不能只把缓存里自报的 identity 复制回去当验证。磁盘映射避免一次装入全部特征，读取性能仍取决于存储、批次整理和传输。 [项目缓存接口](../../src/minigen/data/representation_cache.py) · [P2 验收](../../reports/v2/02_p2_delivery.md)

### **3. 条件丢弃怎样与 CFG 接起来**

每个样本以 $p=0.1$ 的基线概率把 **整条文本特征和对应掩码** 换成 `blank()`。空文本仍保留有效特殊 token；把正常文本特征全置零或者把所有槽位都 mask 掉，定义的是其他输入分布。训练与采样必须一致。文本投影层属于可训练主干，VAE 和 CLIP 参数冻结。

抽查数据时同步看图、原始 Caption、清洗后的 Caption 和有效 token 数。即使语言长度合格，文字仍可能描述不存在的对象。清洗语法和修正事实是两件事。若修改 Caption，需生成新文本/缓存身份，三模型共同使用；不得只给某个模型更准确的条件。

## **三、先做实验登记，再按关卡训练**

### **1. 一份配置要回答哪些问题**

| 配置组 | 必填内容 |
| --- | --- |
| 身份 | run_id、代码提交及未提交 diff 摘要、环境锁、架构规格、数据/缓存/组件摘要 |
| 优化 | 初始 LR、AdamW betas/eps/weight_decay、衰减参数组、warmup/衰减函数、最大更新数 |
| 输入 | 有效 batch、micro-batch、尾批规则、时间分布、噪声/条件丢弃 RNG、数据顺序 |
| 数值 | 参数 dtype、autocast dtype、TF32 策略、实际注意力后端、梯度裁剪、EMA |
| 监测 | 日志频率、验证噪声与时间档、固定验证 Prompt、采样 NFE/CFG、保存和选型规则 |
| 成本与停止 | 墙钟上限、预计费用、磁盘额度、非有限值/失联/质量失败时的处置 |

计划中的 AdamW LR=$10^{-4}$、有效 batch=64、条件丢弃=0.1 是 **试点起点**。正式更新数、EMA 系数、weight decay、warmup 长度等仍需短跑确定，不能补成已经验证的最佳值。配置里的未知项用 `null` 加原因，正式启动前必需项必须确定。固定总更新数 $U$ 后，一个可解释的 warmup+cosine 方案是先用 $U_w$ 次成功更新从低学习率升到 $\eta_{\max}$，再降到预定 $\eta_{\min}$；记录第一个实际更新用的 LR，避免 scheduler 调用顺序差一位。

### **2. 顺序与停止条件**

1. **数据与表示就绪。** 审核 Caption 噪声决策，验证缓存及空条件身份；不根据测试图片修数据。
2. **接口与数学就绪。** 验证时间方向、patch 往返、速度形状、mask、条件路径、梯度和冻结参数。零初始化头会暂时挡住前层梯度，检查时须主动开门或检查更新后的模型。
3. **16–32 张固定监督拟合。** 仅用 train 样本，固定 $z,\epsilon,t,c$ 检查能否拟合；它不能证明模型学会总体生成。接着恢复随机时间、噪声和丢弃策略。
4. **500–1000 步有限短跑。** 在正式 256px 路径上看曲线、固定全量图库、条件干预及失败；步数是诊断范围，不保证此时一定有可辨图片。失败先定位原因，不为凑门槛挑好图。
5. **同硬件资源 pilot。** B0 完成预热、计时、显存与短恢复验证；给出正式预算配置。收费资源及正式 CUDA 长训按用户授权执行。
6. **B0 正式训练。** 保留中间 checkpoint；稳定后让 B1、B2 走相同短跑、恢复、资源门槛，再开展配对训练。
7. **验证选型、冻结测试。** 明确最终 checkpoint、EMA/raw、CFG、NFE 和评分器版本，随后统一测试。

[P3–P5 要求](../MiniGen2重构规划.md) 是工程任务的依据。本课不会替这些任务勾选完成。把每个门槛的命令、日志、配置和结论放在实际 run 记录里，失败的配置也保留。

## **四、一次优化更新里，哪些顺序不能错**

### **1. micro-batch、累积窗口和真实 batch**

micro-batch 是一次前后向看到的样本；梯度累积窗口包含多次前后向；optimizer update 才真正修改参数。单卡每次等长 micro-batch 为 $m$、累积 $a$ 次时，$B_{\mathrm{eff}}=ma$。若 $W$ 张卡各自等长，则 $B_{\mathrm{global}}=Wma$，但 DDP 同步和尾批行为还需单独验证。

如果第 $j$ 个 micro-batch 有 $b_j$ 个样本、平均损失为 $L_j$，整个窗口正确的平均损失是：

$$L_{\mathrm{window}}=\frac{\sum_j b_j L_j}{\sum_j b_j},\qquad \nabla L_{\mathrm{window}}=\sum_j\frac{b_j}{B_{\mathrm{actual}}}\nabla L_j.$$

只有所有 $b_j$ 相同时，才可把每个均值损失简单除以累积次数。现有 `StatefulBatchStream` 会保留每轮短尾批，所以正式训练器必须明确处理窗口尾部。可先取得本窗口各批的索引/长度，再逐批搬数据到 GPU；不要为累积一次性保留所有计算图。跨 epoch 补足窗口、保留短窗口或丢尾批都可以成为协议，但要记录实际样本暴露量和数据游标。

累积通常减少峰值激活内存，但不保证更快，也不保证与大 batch 逐位等价。Dropout、浮点求和顺序和依赖整批统计的模块都可能造成差异；本项目应先用无随机层、固定输入验证梯度权重和优化更新。

### **2. FP32、BF16 与 FP16 分别负责什么**

混合精度让适合的乘法使用低精度，参数及优化器状态可以继续保留 FP32。BF16 的指数范围接近 FP32、尾数较短，通常无需梯度缩放；FP16 的动态范围较窄，常用 `torch.amp.GradScaler("cuda")`。autocast 不等于把整个模型永久转成低精度。

建议先在 CPU FP32 核对逻辑，再在目标 GPU 检查 BF16 支持及数值误差。路径插值、速度标签、误差平方与归约优先 FP32；注意力/归一化是否另需 FP32，要看实际内核与数值证据。不能仅凭设备名字断言所有 BF16 算子都受益。 [PyTorch 混合精度指南](https://docs.pytorch.org/docs/2.14/notes/amp_examples.html)

| 操作 | FP32 / BF16 autocast | 使用 FP16 GradScaler |
| --- | --- | --- |
| 窗口开始 | zero_grad 一次 | 同左 |
| 每个 micro-batch | 加权 loss.backward | scaler.scale(加权 loss).backward |
| 窗口结束 | 检查并裁剪总梯度 | 先 unscale_ 一次，再检查/裁剪 |
| 参数更新 | optimizer.step | scaler.step，再 scaler.update |
| 计数与后续 | 成功更新后推进 LR、EMA、update 数 | 先确认未因 inf/NaN 跳过，再推进 |

不能在累积未结束时更改缩放因子，也不能每个 micro-batch 裁剪一次后声称等价于裁剪总梯度。裁剪为 $g'=g\min(1,\tau/(\lVert g\rVert+\delta))$，这里 $g$ 是完整窗口的梯度；记录的是裁剪前范数。阈值不代替根因排查，持续大量裁剪可能提示 LR、目标尺度或实现问题。

FP16 下 optimizer 的 `step()` 常返回 None，不能用返回值判断是否跳过。单 optimizer、默认 GradScaler 的正常动态缩放下，可比较 update 前后 scale 是否下降来识别溢出跳步；如果使用自定义 scale 或多个 optimizer，需另外设计并测试成功更新事件。日志同时保留 `update_attempt`、`optimizer_step`、`samples_seen`、`skipped_updates`。

### **3. 可移植的单卡更新参考**

下面只定义更新函数，不创建模型或启动训练。它接收 **已经构造好的逐样本 loss 回调**；Flow、条件丢弃和 cache 适配按第二章由工程层接入。窗口是 CPU 数据批列表，回调负责逐批传输和前向，返回 `[b]`；总样本权重在更新函数中处理。它覆盖 CPU FP32 与 CUDA FP32/BF16，不暗含 FP16 跳步处理。

回调按 FP32 构造路径和标签，模型前向由外层 autocast 选择算子精度；最后显式把预测与标签转 FP32 后做差、平方和元素平均。仅在已经算完的低精度 loss 上调用 float()，无法补回之前的舍入。回调不要提前计算 loss，否则会把多个计算图同时留在显存。它应在被调用时才前向。此参考逐批检查有限值会引入同步，适合验收与排错；生产计时若降低检查频率，须单独登记。LR/EMA 在成功返回后由外层各推进一次。调用方预先核对 optimizer 参数集合，并避免在外部嵌套另一种 autocast；所有 optimizer 参数均未收到梯度时，本函数报错，防止出现实际未更新却推进 LR/EMA 的假成功。

In [1]:
import math
import platform
from collections.abc import Callable, Sequence
from contextlib import nullcontext
from typing import Literal

import torch
from jaxtyping import Float
from torch import Tensor


def update_window[BatchT](
    batches: Sequence[tuple[BatchT, int]],
    per_sample_loss: Callable[[BatchT], Float[Tensor, " b"]],  # noqa: F722
    optimizer: torch.optim.Optimizer,
    *,
    device_type: Literal["cpu", "cuda"] = "cpu",
    use_bf16: bool = False,
    max_grad_norm: float = 1.0,
) -> dict[str, float | int]:
    """完成单卡 FP32/BF16 的一个实际样本加权更新窗口。

    Args:
        batches: CPU 批对象及各自真实样本数，窗口开始前确定。
        per_sample_loss: 执行传输和前向，返回 [b] 逐样本 FP32 loss。
        optimizer: 已核对参数集合的优化器；由调用方负责模型 train 模式。
        device_type: cpu 或 cuda；调用方确保模型与回调输入在同一设备。
        use_bf16: 仅支持经过验收的 CUDA BF16 autocast 路径。
        max_grad_norm: 正且有限的总梯度裁剪阈值。

    Raises:
        ValueError: 窗口、批长度、精度组合、loss 形状/dtype 或裁剪阈值不合法。
        FloatingPointError: 出现非有限的 loss。
        RuntimeError: CUDA BF16 不可用、非有限梯度、loss 未连接 optimizer 参数、反传或优化器错误。
        Exception: 回调异常原样传播；无法保证 optimizer.step 内部失败时回滚。

    Results:
        dict: 本窗口的实际样本数、样本平均 loss 与裁剪前梯度范数。

    副作用:
        清理并累积梯度，成功时调用一次 optimizer.step；不更新 LR 或 EMA。
        回调可消耗训练 RNG；失败后按边界 checkpoint 重放数据。
    """
    if not batches or any(n <= 0 for _, n in batches):
        raise ValueError("窗口必须包含正样本数的批次")
    if device_type not in ("cpu", "cuda"):
        raise ValueError("此参考仅覆盖 CPU/CUDA")
    if not math.isfinite(max_grad_norm) or max_grad_norm <= 0:
        raise ValueError("max_grad_norm 必须正且有限")
    if use_bf16 and device_type != "cuda":
        raise ValueError("本参考的 BF16 路径仅在 CUDA 启用")
    if use_bf16 and not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
        raise RuntimeError("本机 CUDA BF16 不可用")

    total = sum(n for _, n in batches)
    parameters = [p for group in optimizer.param_groups for p in group["params"]]
    optimizer.zero_grad(set_to_none=True)
    mean_loss = 0.0
    try:
        for batch, n in batches:
            context = torch.autocast("cuda", dtype=torch.bfloat16) if use_bf16 else nullcontext()
            with context:
                losses = per_sample_loss(batch)
            if losses.dtype != torch.float32:
                raise ValueError("回调须在平方前转 FP32，返回 FP32 逐样本 loss")
            if losses.shape != (n,):
                raise ValueError("回调必须返回逐样本 [b] loss")
            if not bool(torch.isfinite(losses).all()):
                raise FloatingPointError("loss 非有限")
            # 每例等权，避免短尾批被赋予和完整批相同的总权重。
            weighted = losses.float().sum() / total
            weighted.backward()
            mean_loss += float(weighted.detach())
        if not any(p.grad is not None for p in parameters):
            raise RuntimeError("loss 未连接到 optimizer 的任何参数")
        norm = torch.nn.utils.clip_grad_norm_(parameters, max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
    except Exception:
        optimizer.zero_grad(set_to_none=True)
        raise
    optimizer.zero_grad(set_to_none=True)
    return {
        "samples": total,
        "loss": mean_loss,
        "grad_norm_before_clip": float(norm),
    }


print(f"工程参考已定义 | Python {platform.python_version()} | PyTorch {torch.__version__}")
print("尚未绑定正式模型、数据或训练入口；执行本单元不调用 update_window。")

工程参考已定义 | Python 3.12.14 | PyTorch 2.14.1
尚未绑定正式模型、数据或训练入口；执行本单元不调用 update_window。


### **4. AdamW、学习率与 EMA 怎样配合**

AdamW 的 weight decay 与梯度中的 L2 惩罚分开处理。若决定让偏置、归一化参数和某些嵌入免衰减，应在三模型中采用同一规则，并记录参数名和数量；不要因为参数名称不同，意外使某个模型的调制层完全不衰减。Adam 的一阶/二阶矩、optimizer step 计数都是恢复训练的一部分。 [AdamW 论文](https://arxiv.org/abs/1711.05101)

EMA 是参数的指数滑动平均，这里取 $0<\beta<1$：

$$\bar\theta_k=\beta\bar\theta_{k-1}+(1-\beta)\theta_k.$$

从初始模型拷贝 EMA，冻结其梯度，通常在每次 **成功更新之后** 用新参数更新。展开后，$j$ 步前参数的权重与 $\beta^j$ 成正比，半衰期为 $k_{1/2}=\ln(0.5)/\ln\beta$。因此固定 $\beta$ 表示按更新次数平滑；若有效 batch 改变，相同更新数对应的图像数也改变。要按相同样本半衰期 $H$ 比较，可用 $\beta=2^{-B_{\mathrm{actual}}/H}$，但这属于需要统一冻结的另一种 EMA 约定。

EMA 用 `no_grad` 和名字/形状匹配更新参数；整数 buffer 直接复制，浮点 buffer 的策略由模块语义决定。不能随意对所有 `state_dict` 值做插值。验证使用 EMA 或 raw 要在日志里标明；保存两份，恢复训练继续优化 raw。 [DiT 官方训练脚本的 EMA](https://github.com/facebookresearch/DiT/blob/main/train.py)

## **五、把显存、吞吐与总费用拆开看**

### **1. 显存并不只由参数量决定**

以 FP32 参数、梯度、Adam 两份矩状态、FP32 EMA 为例，主要持久张量约为 $4P+4P+8P+4P=20P$ 字节。30M 参数对应约 600 MB 十进制；这还没算激活、算子 workspace、CUDA context、allocator、数据批、编译缓存及 foreach 优化器临时量。它是组成核算，不是 GPU 需求预测。

标准多头注意力的打分张量为 `[b,h_a,n_q,n_k]`，单层存储约 $b h_a n_q n_k s$ 字节，其中 $s$ 为元素字节数。多层反向保留量取决于内核与重算策略。增加分辨率增加 token 数，模型参数本身通常不变。文本 padding 的 mask 能屏蔽语义，但密集内核未必跳过对应计算。

单个注意力的 QK 和 AV 两次矩阵乘法约有 $4b n_q n_k d$ FLOPs（一次乘加计 2，忽略 softmax 和投影）。同宽度下，这能解释联合注意力为何处理更多位置；整网还需加 QKV/输出投影、MLP、调制、反向和 optimizer。只报 $6P$ 乘 token 数会遗漏架构和序列交互差异。profiler 的 FLOPs 统计也可能漏算 fused 算子，报告覆盖范围，不填一个看似精确的总数。

### **2. SDPA、FlashAttention 与重算**

`scaled_dot_product_attention` 是 PyTorch 算子接口，根据设备、dtype、形状、mask 和版本选择后端。FlashAttention 是其中一类高效精确注意力算法，通过分块和减少显存往返降低开销；它不把密集注意力的两两交互变成线性复杂度，也不保证浮点结果逐位一致。 [SDPA 文档](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.scaled_dot_product_attention.html) · [FlashAttention 论文](https://arxiv.org/abs/2205.14135)

先对同一 Q/K/V、mask 比较显式实现与 SDPA 的输出和输入梯度；关闭 dropout，使用与精度匹配的容差。随后在真实训练形状下测 forward+backward，确认实际后端。不能用无 mask 的快速路径成绩描述正式有 mask 的路径。函数式 SDPA 在 eval 时仍按 `dropout_p` 参数执行，评估应显式设为 0。

activation checkpointing 在反向重算部分前向，以时间换激活内存；它与写磁盘的训练 checkpoint 不同。可从 DiT Block 边界试用 `checkpoint(block, ..., use_reentrant=False)`，检查 RNG、设备移动和有状态副作用。不要在被重算区域里更新 EMA、写日志或推进数据游标。前向与重算不一致会破坏梯度。 [PyTorch 重算文档](https://docs.pytorch.org/docs/2.14/checkpoint.html)

### **3. 按瓶颈安排优化顺序**

先记录一段未优化的真实流水线，再依次检查：

1. 若 GPU 经常等待，测缓存读取、collate、CPU→GPU 传输。逐渐调整 worker、pin_memory 和预取；`non_blocking=True` 本身不保证传输与计算重叠。
2. 若 OOM 出现在 forward/backward，减 micro-batch、用累积补 batch；再试 BF16、SDPA 和 Block 重算。若在首个 optimizer.step 才 OOM，检查刚分配的 Adam 状态和 foreach 临时空间。
3. 若每步内存持续增长，检查日志是否存了带图 Tensor、是否遗漏 detach、是否保留全部预测。`empty_cache()` 不能释放仍被引用的张量，也不应每步调用。
4. 数值和流水线稳定后才试 compile，单独记录编译首轮开销与稳态收益。短 run 可能无法摊销编译成本。

默认单 GPU。DDP 复制模型并分摊 batch，不能自动解决单份模型状态放不下的问题；ZeRO/FSDP 进一步分片状态，有通信与恢复复杂度。只有单卡证据表明需要这些技术，才另开任务。DDP 累积要注意 `no_sync` 覆盖前向与反向，且跨 rank 样本数、loss 归一化、最后一批和验证汇总必须匹配。

### **4. 怎样测一个可相信的秒数**

正式设备上的 pilot 建议先做 50–200 次预热/计时更新，具体数量以稳态为准。CUDA 是异步执行，主机调用返回不等于 GPU 做完。端到端计时在窗口起止同步，用墙钟覆盖取数、传输、forward、backward 和更新；GPU 内核时间可另外用 CUDA Events。不要每步同步后把它当成最优吞吐。

固定 GPU 型号、驱动/PyTorch/CUDA、dtype、batch、累积、后端、重算、编译和测量窗口。重置峰值计数后记录 `max_memory_allocated` 与 `max_memory_reserved`；前者是张量分配峰值，后者含 allocator 保留池，不相加。吞吐用实际处理样本数除以窗口墙钟，逐样本 `loss.item()` 与频繁保存的同步成本另记。 [CUDA 内存与异步行为](https://docs.pytorch.org/docs/2.14/notes/cuda.html) · [PyTorch Profiler](https://docs.pytorch.org/docs/2.14/profiler.html)

$$T_{\mathrm{run}}\approx U_{\mathrm{attempts}}\,t_{\mathrm{attempt}}+N_{\mathrm{val}}t_{\mathrm{val}}+N_{\mathrm{save}}t_{\mathrm{save}}+T_{\mathrm{startup}}.$$

$$C_{\mathrm{estimate}}=\frac{T_{\mathrm{billed}}}{3600}\,r_{\mathrm{hour}}+C_{\mathrm{storage}}+C_{\mathrm{transfer}}.$$

这里的尝试次数包含因溢出跳过参数更新的窗口，平均尝试耗时取自 pilot；若目标按成功更新数制定，应计入实测跳步率带来的额外尝试。区分纯训练时间、作业墙钟和平台计费时间。费用表列单价来源、时间、计费粒度及重试预留；不要把未执行的训练写成实际费用。本期旧预算 100–300 元是历史估计，需由真实 pilot 更新。

## **六、保存与恢复要让下一批也接得上**

### **1. checkpoint 是优化过程的快照**

已知模型结构、冻结组件、预处理与采样规则时，主干权重可以恢复推理；单靠这份权重无法恢复同一条训练轨迹。完整 checkpoint 至少包含：

| 内容 | 恢复用途 |
| --- | --- |
| raw 模型、EMA、模型配置 | 参数、结构与采样身份 |
| optimizer、LR scheduler、GradScaler（若有） | 动量、二阶矩、实际 LR 与缩放因子 |
| attempt/update/样本数/跳步数 | 区分读取过的数据和成功优化 |
| Python、NumPy、Torch CPU/CUDA、各独立 Generator RNG | 重建顺序、噪声、时间、条件丢弃 |
| 数据身份、排列、epoch、cursor、窗口策略 | 找到下一份样本与正确尾批 |
| 配置、代码/环境、组件/缓存摘要 | 拒绝不兼容的直接续训 |

优先在成功 update、LR/EMA 都完成后的 **窗口边界** 保存，这时没有需要恢复的半窗口梯度。若中途出错，恢复最近边界并重新走剩余数据；不能把预取后前移的 cursor 配上更新前的模型。保存到同文件系统临时文件，flush/必要时 fsync 后原子替换，最后公布索引与摘要。若异步保存，要拷贝一致的状态快照，避免后台读到仍在变化的参数。

### **2. 恢复的正确顺序与验证方法**

先核对 identity 并创建模型、optimizer、scheduler 和数据流；加载其 state，再按实现检查 optimizer 张量设备；最后恢复所有 RNG 和游标，在下一批之前不额外消耗训练随机数。scheduler 构造与 load 顺序需按所用 PyTorch 文档核对，恢复后直接比较实际 param_groups LR。只加载可信本地 checkpoint，`torch.load` 的对象反序列化与 `weights_only` 限制需要明确处理。 [PyTorch 保存恢复指南](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)

验证方法是同一环境下做两条分支：连续运行到 $K+J$；运行 $K$ 后保存，在新进程恢复并再跑 $J$。逐步比较 sample ID、噪声/时间摘要、loss、LR、raw/EMA、optimizer 和 RNG。先要求确定的 CPU 小路径精确对照，再给目标 GPU 设数值容差；不能只比较保存前后的模型输出。

现有 `StatefulBatchStream` 支持单进程确定数据访问的排列与 cursor 恢复。增加多 worker、随机增强和预取后，worker RNG、预取队列、sampler 已请求位置与真正消费位置可能不同，旧结论不自动成立。CUDA 非确定算子和跨设备浮点差异也使「同种子」不等于逐位复现。 [项目数据流](../../src/minigen/data/stateful.py) · [PyTorch 可复现性](https://docs.pytorch.org/docs/2.14/notes/randomness.html)

## **七、训练中看什么，发现问题后按什么顺序查**

### **1. 日志与验证要各有职责**

每个尝试窗口记录状态、实际 batch、样本累计、raw loss、LR、梯度范数、时间和跳步数，成功后另记实际 update 编号；按窗口聚合吞吐与显存。固定周期保存三类诊断：

- **验证速度误差。** 对相同验证 latent、噪声、时间档和文本计算逐样本 loss；先样本平均再按统一规则汇总时间档。低噪声/高噪声两端分开看，避免某段容易任务掩盖另一段。
- **条件反事实。** 固定 $z_t,t,u$，比较正确、打乱和空条件；用合理匹配的替换避免批中重复 Caption 让「打乱」仍然正确。它检验条件是否产生有益影响，不能替代生成评测。
- **固定采样图库。** 固定 Prompt、noise seed、sampler、CFG、NFE、EMA/raw。展示完整网格与失败，不能每次换 seed 找更好图。

验证应使用 `eval()`、`inference_mode/no_grad` 和独立 RNG，结束后恢复训练模式；固定验证种子不能重置训练 RNG。多次可视化若消耗训练随机流，会改变后续训练，导致恢复与对照不一致。

### **2. 先定位第一处异常，再决定改哪项**

| 症状 | 优先检查 | 有证据后采取的动作 |
| --- | --- | --- |
| 首步 NaN/Inf | 缓存有限值、目标尺度、全遮 mask、低精度 softmax、RoPE 维度 | 单批 FP32 复验，定位首个非有限层；修输入/算子再谈降 LR |
| 梯度一直为零 | 参数 requires_grad、optimizer 参数表、detach、零头/门阶段 | 检查输出头首步梯度、更新后上游路径；不能把初始化暂时零梯度判成故障 |
| loss 下降，图片像噪声 | 时间方向、速度/噪声输出混用、scale 重复、采样端点 | 同一 checkpoint 用已验证采样器，检查解码重建与速度范围 |
| 图好看但不听文字 | Caption 错配、mask、空条件比例、冻结位置、错条件探针 | 逐 ID 查图文配对，验证条件梯度，再评估条件数据覆盖 |
| 训练低而验证高 | 近重复、训练集记忆、样本不足、正则与选型次数 | 最近邻检查、固定验证协议；未经证据不归咎于架构 |
| 恢复后曲线突变 | optimizer/EMA/LR/RNG、数据 cursor、缓存/组件变化 | 比较第一处分岔，做短连续/恢复对照 |
| 改优化后速度没提高 | 数据等待、回退内核、首次编译、过多同步 | profiler 定位并重测真实窗口 |
| 更强 CFG 更差 | 过强向量外推、无条件分支不足、步长/时间表 | 验证集看质量与遵循曲线，共同选择 CFG 和 NFE |

VAE 重建本身已经丢失的细字或细线，不能要求生成主干无成本补回。先看同域真实图的重建上限，再评估主干；更换 VAE 会重置表示条件，需要统一版本与重做比较。

## **八、让 B0、B1、B2 的比较回答同一个问题**

### **1. 固定协议与允许变化的部分**

三个模型均从随机初始化训练约 30M 参数主干，目标差异约在 ±10% 内，但要同时报告实际参数和两两差距。冻结相同数据、Caption、VAE/CLIP、Flow、有效 batch、优化器、LR/EMA 规则、验证与采样；每个模型分别记录宽度、头数、层数、双流/单流比例、位置和可见性。相同层数不代表相同参数，也不代表相同 FLOPs。

若同时改了 RoPE、mask、宽度和模态投影，结论是两套架构配方的比较。想隔离「共享还是独立投影」的作用，需要另做只改变对应因素的消融。正式首轮 B0/B1/B2 各 seed=0；额外 seed=1、10M/50M 是结果与预算明确后的选项。

### **2. 同步数与同计算预算是两条横轴**

同更新次数、同 batch 比较数据暴露量；同 GPU 小时或估算 FLOPs 比较资源利用。在同设备和稳定配置下累计每个 run 的实际训练时间，使用预定的中间 checkpoint 画质量—时间曲线。时间预算边界取最后一个不超过预算的 checkpoint，报告实际差距；不要从未来更好 checkpoint 插值出不存在的生成样本。

Abra 在大规模训练计算范围里研究模型容量、训练数据和质量的关系。本项目借鉴其 **固定总计算预算再比较容量/训练长度** 的方法；论文拟合出的最优比例依赖它的数据、架构、目标和预算，不能直接给 ArtBench 30M 模型规定训练轮数。 [Abra 论文](https://arxiv.org/abs/2608.17286v1)

$$N_{\mathrm{seen}}=\sum_{k=1}^{U_{\mathrm{attempts}}} B_k,\qquad E_{\mathrm{exposure}}=N_{\mathrm{seen}}/N_{\mathrm{train}}.$$

样本暴露数包含跳步时读过的 batch，而成功更新另计。独立验证采样耗时在总作业成本中单列。论文里的一般 scaling 趋势和这个小数据集上的实测排名需要分别陈述。

## **九、评测先定义问题，再生成图片**

### **1. 两套集合不能互相替代**

集合 A 用于分布质量：从冻结 ArtBench 测试划分抽取约 1,000–2,000 张、覆盖风格且与训练组隔离的参考图，并定义相匹配的条件采样规则生成图片。它衡量「生成分布是否接近指定真实分布」，不要求一张生成图复制对应真实图。真实参考图自己的 VAE 重建也可作为 **表示诊断**，另报分数，不能当理论最低距离。

集合 B 是条件题单：计划 ID 80、Composition-OOD 40、Paraphrase 40，共 160 条，每条 2 个噪声 seed，即每模型 320 张。另建 80–120 条验证 Prompt 用于选择 checkpoint、CFG、NFE。320 张条件题单的概念分布不一定代表 ArtBench，不能直接拿它与全域真实集算一个距离后声称是整体生成质量。

组合外需要训练前定义禁入组合，并检查清单和 Caption 是否支持这种隔离。当前自然绘画 Caption 有噪声，若无法确认某组合在训练中没出现，就应称为 **组合能力探针**，不能只凭题目新颖就写 OOD。改写题应绑定同一个语义 family，尽量用同噪声成对生成；本课的要求是制定规则，不替尚未构建的题单声称冻结完成。

### **2. Prompt 与生成记录要能对回同一张图**

| 清单 | 必需字段与用途 |
| --- | --- |
| prompt.jsonl | prompt_id、family_id、split、英文文本、主体/属性/数量/关系要求、适用指标、来源、审核和版本 |
| generation.jsonl | run/checkpoint/EMA 身份、prompt_id、noise_seed、sampler、时间表、CFG、NFE 各口径、dtype/设备、路径/文件哈希、status、耗时 |
| judgement.jsonl | generation_id、评分器版本、每题 raw answer/解析状态/得分、人工审核、适用分母、失败原因 |
| metrics.json | 输入清单哈希、实际 n、特征/预处理/核版本、点估计、区间和计算方法 |

图片文件名不能成为唯一的实验身份。失败记录保留原请求、状态和原因；修复基础设施后重试采用相同请求并保留两次记录，不能偷偷换 seed。schema 是本篇的工程契约，后续 P5 才实现正式写入器。本课不生成假样本清单。

## **十、KID、CMMD 与条件评分究竟测了什么**

### **1. 从特征分布到 MMD**

把真实图与生成图分别经过固定特征提取器得到 $x_i,y_j$。核函数 $k$ 衡量两个特征的相似性。最大均值差异 MMD 用「各分布内部相似度」减去「跨分布相似度」：

$$\operatorname{MMD}^2(P,Q)=\mathbb E[k(X,X')]+\mathbb E[k(Y,Y')]-2\mathbb E[k(X,Y)].$$

其中 $X,X'$ 独立来自 $P$，$Y,Y'$ 独立来自 $Q$。假设两组样本各自独立同分布、两组也相互独立，样本数为 $n,m\ge2$，去掉自配对的无偏 U 估计是：

$$\widehat{\operatorname{MMD}}_u^2=\frac{\sum_{i\ne j}k(x_i,x_j)}{n(n-1)}+\frac{\sum_{i\ne j}k(y_i,y_j)}{m(m-1)}-\frac{2\sum_{i,j}k(x_i,y_j)}{nm}.$$

按同一参考图的 Caption 配对生成、共享 Prompt 的多 seed 等设计可能引入相关性，完整评测协议的无偏性还需另行分析。总体非负不意味着有限样本的无偏估计非负；略负的 KID 可能来自抽样波动，不该强行截为零。特征提取器选定了被比较的信息，特征相似不保证所有人可见错误都会被发现。

### **2. KID 和 CMMD 要固定到实现**

KID 常用 Inception 特征和三次多项式核 $k(x,y)=(x^\top y/d+1)^3$，以 MMD 的无偏估计衡量分布差异。记录 Inception 版本、输入尺寸、像素范围、特征层、子集大小、重复次数和种子。子集得分的标准差描述抽样变化，不能直接当模型总体优劣的置信区间。 [KID 原论文](https://arxiv.org/abs/1801.01401v5)

CMMD 使用 CLIP 图像特征和高斯核，减少对 FID 高斯拟合等假设的依赖。它比较的是 **两组图像**，不会自动检查每张图是否符合对应 Prompt。官方代码当前的默认特征为 CLIP ViT-L/14@336，`distance.py` 使用含对角项的 biased V 估计、$\sigma=10$、显示缩放 1000：

$$\widehat{\operatorname{MMD}}_v^2=\frac{\sum_{i,j}k(x_i,x_j)}{n^2}+\frac{\sum_{i,j}k(y_i,y_j)}{m^2}-\frac{2\sum_{i,j}k(x_i,y_j)}{nm},\quad k(x,y)=\exp\left(-\frac{\lVert x-y\rVert^2}{2\sigma^2}\right).$$

不能直接把上面的 KID U 公式换个编码器就标成「官方 CMMD」。冻结所用代码 revision、核和特征归一化，报告实际估计器；改变特征维度、归一化或核带宽都会改变数值尺度。 [CMMD 论文](https://arxiv.org/abs/2401.09603) · [官方实现](https://github.com/google-research/google-research/tree/master/cmmd) · [核估计代码](https://github.com/google-research/google-research/blob/master/cmmd/distance.py)

KID/CMMD 在绘画域的解释需要视觉审核辅助。生成失败时，条件成功率分母仍包含失败；分布距离只能对可解码图像计算，必须同时报告计划数、成功数和失败率。若不同模型有效集合差异明显，不能用幸存图片的距离单独宣布优胜。

### **3. CLIPScore、VQA 和人评互相补充**

CLIPScore 用图文特征相似度估计语义匹配。作者实现默认采用 $2.5\max(\cos(e_I,e_T),0)$，其中 $e_I,e_T$ 是归一化图文特征，另外固定文本前缀与预处理，不能把任意 CLIP cosine 直接标成同一分数。若使用项目 CLIP-B/32 算自定义 cosine，应明确命名，并说明与训练条件编码器相关带来的表示偏好。 [CLIPScore 论文](https://aclanthology.org/2021.emnlp-main.595/) · [作者实现](https://github.com/jmhessel/clipscore)

VQA 先把要求拆成可判断的问题，再逐项判对象存在、颜色绑定、数量和关系。分别问「有红色」「有圆形」并不能证明「红色的是圆形」；关系题还要确定主体和参照物。对不适用的属性不计入该项分母，对生成失败记失败，对评测器解析失败保留状态并人工复核。整图成功要求所有适用条件同时通过。

项目计划要求至少约 100 张真实图和约 100 张生成/人为退化图进行人工校准；加入已知正反条件、数量错与绑定错，测分类别误判率和漏判率。VLM 的自己解释不能当人工真值，人工评分还需匿名、打乱、统一 rubric，必要时双人复核分歧。绘画域不适合直接复用旧几何解析器。 [T2I-CompBench 初版任务设计](https://arxiv.org/abs/2307.06350v1)

LPIPS 可用于同 Prompt 多 seed 的视觉差异，但「差得很大」可能只是崩图。先报告有效率和条件成功率，再在明确的有效子集上报告多样性，并公开筛选分母。最近邻同时用像素/感知特征和人工对照检查记忆，不以一个距离阈值证明没有复制。

## **十一、采样消融按模型求值次数记账**

### **1. Euler、Heun 与 CFG 的三个计算口径**

沿本项目 $t:0\to1$ 积分，Euler 一步 $z_{k+1}=z_k+\Delta t\,v_k$，每区间求值一次。标准显式 Heun 先预测端点，再用首尾速度平均，通常每区间两次；若最后一段改 Euler，总次数是 $2K-1$，必须标明这种变体。训练时间随机采样通常不包含精确端点，采样器若在 $t=1$ 求值，还要验证网络/时间嵌入边界。

CFG 定义为 $v_s=v_\varnothing+s(v_c-v_\varnothing)$。$s=1$ 时直接用 $v_c$，可以跳过无条件分支；$s>1$ 时需要两支。两支拼成一个 $2b$ batch 只减少 Python 调用次数，没有把算术成本变为一支。

- **field_nfe** 是 ODE 对组合后向量场的求值次数。
- **branch_evals_per_image** 是每张图条件/无条件网络评估总次数。
- **forward_calls** 是实际执行的主干调用次数，还受 batching、切片和缓存影响。

以标准 Heun 为约定，field_nfe=20 对应 10 个区间；Euler 对应 20 个区间。两者必须使用相同的时间方向和明确的时间表。相同 field_nfe、不同 CFG 的算量仍可能不同。缓存文本编码不等于缓存每层文本 K/V，S5/S6 的双向条件路径不能直接套用 Qwen 的跨步前缀缓存。

### **2. 先小范围验证，再冻结主评测**

验证集候选为 CFG=1/2/3、field_nfe=10/20/40、Euler/Heun。完整三架构网格共有 $3\times3\times3\times2=54$ 个配置，每配置 320 张就达 17,280 张，还不含集合 A 和训练中验证。先用预登记的小验证子集筛选，再在完整验证集复核；主测试仅跑冻结配方及预先确定的重点消融。参数搜索本身计入研究总成本。

推理计时分别报告 DiT 迭代、VAE 解码、文本编码和完整端到端。预热后同步，固定分辨率、batch、种子、精度和条件缓存状态；首次加载/编译另报。质量—NFE 曲线和质量—延迟曲线回答不同问题，不能互相替代。

In [2]:
def sampling_budget(
    field_nfe: int,
    sampler: Literal["euler", "heun"],
    cfg_scale: float,
    cfg_batched: bool = True,
) -> dict[str, int]:
    """按标准 Heun 和统一 CFG 约定计算单个采样批的理论调用账本。

    Args:
        field_nfe: 组合速度场求值次数，必须为正整数。
        sampler: euler 或每个区间完整两次求值的 heun。
        cfg_scale: 支持本项目候选的有限值 s>=1。
        cfg_batched: 双分支是否在同一次 forward 中拼 batch。

    Raises:
        ValueError: NFE、采样器、CFG 不合法，或 Heun 收到奇数预算。

    Results:
        dict: intervals 为积分区间数，field_nfe 为场求值数；
            branch_evals_per_image 为每图分支评估数；
            forward_calls_per_batch 为不做额外切片时的调用数。
    """
    if type(field_nfe) is not int or field_nfe <= 0:
        raise ValueError("field_nfe 必须是正整数")
    if sampler not in ("euler", "heun"):
        raise ValueError("未知采样器")
    if not math.isfinite(cfg_scale) or cfg_scale < 1:
        raise ValueError("此账本只接受有限 CFG>=1")
    if sampler == "heun" and field_nfe % 2:
        raise ValueError("标准 Heun 每区间两次求值，需要偶数预算")
    branches = 1 if cfg_scale == 1 else 2
    return {
        "intervals": field_nfe if sampler == "euler" else field_nfe // 2,
        "field_nfe": field_nfe,
        "branch_evals_per_image": field_nfe * branches,
        "forward_calls_per_batch": (field_nfe if cfg_batched else field_nfe * branches),
    }


print("采样账本函数已定义；实际 profiler 仍需核对 forward 调用与批大小。")

采样账本函数已定义；实际 profiler 仍需核对 forward 调用与批大小。


## **十二、怎样从结果得出可信结论**

### **1. 配对比较与不确定性**

两模型对同一 Prompt、noise seed 使用相同评价器，可以先算逐项差值 $d_i=s_{A,i}-s_{B,i}$，再汇总。若同一语义 family 包含原句、改写和多个噪声，它们相关；做 bootstrap 时应整组抽 family，保留组内两模型的配对。不要把 320 张相关样本都当独立重复。

训练 seed 是更高一层随机性。每个训练 run 独立列出指标、成本和失败；两次 run 只给粗略波动信息。对单一 checkpoint 做上千次图像 bootstrap，仍不能替代重新训练多个 seed。区间表示所定义抽样单位下的不确定性，不能覆盖数据偏差、评分器偏差和未做的训练重复。

不同 family 包含不同数量的图片时，预先确定按图平均还是先 family 平均再平均；ID、OOD、改写分别报告。对分布指标的 bootstrap 还要同时考虑真实集和生成集，并重新计算统计量；KID 的随机子集波动与这种区间含义不同。

### **2. 报告图表与失败要对得回原始记录**

最终至少保留质量—更新数、质量—累计 GPU 时间、质量—NFE/延迟三类曲线；标出实际参数、EMA/raw 和误差范围。不能把不同 run 的最优点连成一条虚构训练轨迹。表中的未知 FLOPs/成本写未测，不填推测数。

失败分组至少覆盖对象缺失、数量错、绑定错、关系错、纹理/结构缺陷和解析失败；同一图可有多个标签。全量图片索引对回 generation.jsonl 和 checkpoint 摘要。结论采用「在这个数据、规模、训练预算和评测条件下观察到……」，同时说明没有控制的架构因素。小模型训练结论不能据此反推原版 Qwen/Hunyuan 的优劣。

## **十三、把知识落实到你接下来的实际工作**

### **1. 从 P3 到最终报告的操作单**

| 顺序 | 你需要理解或决定 | Agent 可以完成的工程 | 验收后留下什么 |
| --- | --- | --- | --- |
| P3-01/02 | Flow/Condition/速度输出，B0 张量流 | 接正式缓存、统一接口、配置与参数统计 | 数据到损失的接口说明 |
| P3-03/04 | 为什么数学、mask、更新和恢复测试会失败 | 实现训练器/采样器入口及独立测试 | 真正可执行命令和短恢复证据 |
| P3-05 | loss 与生成诊断的区别 | 执行小样本、随机短跑、固定图库 | 失败定位和条件收益 |
| P3-06/P4-01 | 成本和质量是否值得继续 | 测目标 GPU、汇总预算与停止规则 | 审核后的 B0 长训配方 |
| P4-02 | 根据验证证据解释收敛 | 运行、保存、定期诊断和恢复 | B0 中间/最终 checkpoint |
| P4-03…07 | 单流/混合差异及可比较范围 | 复用训练器验收并执行 B1/B2 | 三模型统一日志与成本 |
| P5 | 指标限制、消融假设和结论 | 生成全量、评分校准、图表与报告 | 真实矩阵、区间、失败与复现 |

现在可运行的是项目现有诊断，例如 `uv run --locked --offline --no-sync minigen doctor --device cpu --json`。正式 train/sample 命令要等 P3 实现并验收后写入 run；不要照抄规划中的示意命令期待它已经工作。进入训练前先看 [当前交接](../../reports/v2/00_handoff.md)，本篇状态快照会随工程推进变旧。

第一次实操按表逐项完成。遇到错误先保存样本 ID、配置、日志和最小失败输入；一轮只修改有证据指向的因素。常规排错和记录交给 Agent，你重点解释目标、核心数据流、实验假设与结果。

### **2. 三项个人验收与参考答案**

<details><summary>怎样说明一个“有效 batch=64”的 run 实际做对了？</summary>

查看每个更新窗口的真实样本数、尾批规则、loss 权重、zero_grad 和 optimizer.step 次数；用固定输入对比大 batch 与累积梯度，再验证调度器和 EMA 只随成功更新推进。配置写 64 本身不是证据。

</details>

<details><summary>恢复后 loss 看起来差不多，就能继续正式比较吗？</summary>

还不够。要检查相同环境下下一批 ID、噪声、时间、条件丢弃和 LR，以及连续/恢复分支的 raw、EMA 和 optimizer 状态。容差必须预先明确；否则恢复可能变成一条新训练轨迹，应记录为分叉 run。

</details>

<details><summary>B1 的 CLIPScore 高、CMMD 低，能宣布比 B0 更好吗？</summary>

先核对样本集合、失败率、特征与评分器版本、参数/计算预算和训练重复。再看对象/数量/关系、人工质量和全量失败。指标分别衡量图文相关与特征分布，不能覆盖所有能力；若排序随 seed 或预算变化，按实际证据陈述。

</details>

## **十四、资料导航与本篇核验范围**

### **1. 按问题阅读官方来源**

| 资料 | 重点读什么 | 回到本课解决什么 |
| --- | --- | --- |
| [Flow Matching](https://arxiv.org/abs/2210.02747) | 第 3 节的条件回归与概率路径 | 目标、时间方向和边际速度 |
| [DiT 官方 train.py](https://github.com/facebookresearch/DiT/blob/main/train.py) | optimizer/EMA、冻结 VAE、保存路径 | 借鉴职责划分；其 DDPM 目标和本课 Flow 分开 |
| [PyTorch AMP](https://docs.pytorch.org/docs/2.14/notes/amp_examples.html) | accumulation、unscale、gradient clipping | 更新顺序与跳步 |
| [SDPA](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.scaled_dot_product_attention.html) 与 [FlashAttention](https://arxiv.org/abs/2205.14135) | mask、dropout、后端与 IO | 数值等价与真实性能 |
| [重算](https://docs.pytorch.org/docs/2.14/checkpoint.html) 与 [可复现性](https://docs.pytorch.org/docs/2.14/notes/randomness.html) | use_reentrant、RNG、非确定性 | 激活节省与恢复边界 |
| [Abra](https://arxiv.org/abs/2608.17286v1) | 固定训练计算预算的比较方法 | 容量/训练长度的取舍 |
| [KID](https://arxiv.org/abs/1801.01401v5)、[CMMD](https://arxiv.org/abs/2401.09603) | 特征、核和估计方法 | 分布指标的定义和限制 |
| [CLIPScore](https://github.com/jmhessel/clipscore)、[T2I-CompBench](https://arxiv.org/abs/2307.06350v1) | 实际评分处理、绑定/关系题 | 条件遵循与人工校准 |

资料核对日期为 2026-11-08。PyTorch 链接按本机 2.14 系列固定；正式 GPU 环境若变更，核对对应版本的 API 和行为。外部源码的 main/master 可能变化，正式实验须锁实际用到的 revision。本文公式由定义展开，工程代码为本课独立参考，引用用于核对机制与约定。

### **2. 阅读完成与工程完成分别登记**

Notebook 可从干净内核执行；它只导入库、定义工程参考并检查归一化/NFE/EMA 的算术约定，不读取正式缓存、不初始化生成模型。实际训练函数的集成、GPU 性能、恢复与生成质量仍由 P3–P5 验收。下面的输出记录本课代码定义和独立检查，不能视为已训练的证据。

In [3]:
# 纯算术/自动微分检查，不创建生成模型或训练数据集。
from fractions import Fraction

sizes = (3, 1)
assert sum(Fraction(n, sum(sizes)) for n in sizes) == 1
# loss_j = n_j 个相同单项组成的平均，比较总体梯度与分批加权梯度。
theta = torch.tensor(0.25, dtype=torch.float64, requires_grad=True)
coefficients = torch.tensor([1.0, 2.0, 4.0, 7.0], dtype=torch.float64)
direct = torch.autograd.grad(((theta - coefficients) ** 2).mean(), theta)[0]
accumulated = torch.zeros_like(theta)
for chunk in coefficients.split(3):
    part = ((theta - chunk) ** 2).sum() / len(coefficients)
    accumulated += torch.autograd.grad(part, theta)[0]
torch.testing.assert_close(direct, accumulated, rtol=0, atol=0)

assert sampling_budget(20, "euler", 1)["intervals"] == 20
assert sampling_budget(20, "heun", 3)["intervals"] == 10
assert sampling_budget(20, "heun", 3)["branch_evals_per_image"] == 40
assert sampling_budget(20, "heun", 3, False)["forward_calls_per_batch"] == 40
rejected = False
try:
    sampling_budget(19, "heun", 1)
except ValueError:
    rejected = True
assert rejected

beta = 0.99
half_life = math.log(0.5) / math.log(beta)
assert math.isclose(beta**half_life, 0.5, rel_tol=1e-12)
print("通过：不等长微批的样本加权与 FP64 梯度一致。")
print("通过：Euler/标准 Heun、CFG 双分支和奇数预算拒绝规则。")
print("通过：EMA 半衰期关系。")
print("未执行：正式 update_window 集成、CUDA/AMP、恢复训练、模型训练和图像评测。")

通过：不等长微批的样本加权与 FP64 梯度一致。
通过：Euler/标准 Heun、CFG 双分支和奇数预算拒绝规则。
通过：EMA 半衰期关系。
未执行：正式 update_window 集成、CUDA/AMP、恢复训练、模型训练和图像评测。
